## Imports


In [ ]:
#graph generation libraries
import sys
import os
import torch
import pandas as pd
import numpy as np
import random
import scipy
import itertools
import scipy.ndimage as ndimage
import itertools
import scipy.io
import scipy.ndimage as ndimage
import torch
import torch.nn.functional as F
import defaultdict

#plotting
from ipywidgets import interact
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import colorsys
import networkx as nx
import hashlib


from google.colab import drive

# Setup

In [ ]:
# ACCELATOR AND DEVICE INFO
use_gpu = True
cpu = torch.device("cpu")
if use_gpu:
  if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))  # Confirm which GPU
    device = torch.device("cuda")
  else:
      device = torch.device("cpu")
      print("using cpu")
else:
        device = torch.device("cpu")
        print("using cpu")

# DATA DIRECTORIES (EXAMPLE IS MY GOOGLE DRIVE)
drive.mount('/content/drive')
main_directory = '/content/drive/MyDrive/WMH_FC_VE/'
data_directory = '/content/drive/MyDrive/WMH_FC_VE/data/'
graph_directory = '/content/drive/MyDrive/WMH_FC_VE/graphs/'
output_directory = '/content/drive/MyDrive/WMH_FC_VE/'
data_table_filename = "batch1_dti_reformatted_new.csv"
num_rois = 66 #DKT ROIs
num_matter = 4
csf_val=60; wm_val=240; wmh_val=180; gm_val=120

# GRAPH GENERATION PARAMS
subject_ids = None # np array of subject indices (eg, first 5 subjects are array([0,1,2,3,4])
subject_ids = np.arange(0,94);
subject_ids = np.delete(nums,[0,24]);
print("Subject IDs: ")
print(subject_ids)

graph_size = (48,64,64) # size of index map
num_shells = 3
boundary_type = "quantile"
boundaries = [0,40,70,100] # distance boundaries of shells (num_shells + 1)
shell_ang_sectors=[7,8,9] # sqrt of number of solid angles in each shell

min_voxels = 28 # pruning threshold for node size
min_contact = 1 # pruning threshold for edge strength
neighbor_length = 5 # max distance between neighbors' voxels

# base name for graph files
dataname = f'subjectspecific_{graph_size[0]}x{graph_size[1]}x{graph_size[2]}_{num_shells}shells_{shell_ang_sectors}sections_{quantiles}bounds_{min_voxels}min_{min_contact}-{neighbor_length}vcontact'

print(dataname)
print('__________________________________________________________________')

## Graph Generation

### generate_or_load_images_for_graphs
Function that resizes input images (with filenaming convention "{DATABASE_ID}_{STUDY_ID}_full_volume.mat") to appropriate graph size (i.e. size of the index maps), or loads images if already exists. The images used in graph generation are: (1) a GM ROI segmentation termed atlas, and (2) a matter segmentation termed seg

Resizing for numerical images uses trilinear interpolation.
Resizing for categorical images (eg. GM ROIs and matter segmentation) involves one-hot-coding the discrete values into separate images, then trilinearly resizing these seperate "intensity" maps (valued 0-1). Each voxel in the final resized image has the category with the highest "intensity."

In [ ]:
def generate_or_load_images_for_graphs(data_directory=data_directory, data_table=data_table_filename, subject_ids=subject_ids,
                                       num_rois=num_rois, num_matter=num_matter, csf_val=csf_val, wm_val=wm_val, wmh_val=wmh_val, gm_val=gm_val,
                                       nums=nums, graph_size=graph_size, restart=False, filename="images_for_graph_gen.pt"):

   filename = data_directory+filename
   if os.path.exists(filename) and not restart:
      images = torch.load(filename, map_location=device).to(torch.long)
      print(f"Loaded images {filename}")
      return images
   else:
      num_subjects = len(subject_ids)
      data_table = pd.read_csv(data_directory+data_table)
      images = torch.zeros([num_subjects,2,*graph_size], dtype=torch.long, device=device)

      for n in subject_ids:
        row = data_table.iloc[n]
        Id = row.loc["ID"].item()
        mat_contents = scipy.io.loadmat(data_directory+str(Id)+"_full_volume.mat")
        atlas = torch.from_numpy(mat_contents["atlas"]).long()

        # GM ROI atlas resizing
        print(f"Atlas coverage before: {torch.sum(atlas>0)/(atlas.shape[0]*atlas.shape[1]*atlas.shape[2])*100:.4f}%")
        atlas = F.one_hot(atlas, num_classes=num_rois + 1)
        atlas = atlas.permute(3, 0, 1, 2).float().unsqueeze(0)
        atlas = F.interpolate(atlas, size=graph_size, mode="trilinear", align_corners=False)
        atlas = torch.argmax(atlas, dim=1).squeeze(0)
        print(f"Atlas coverage after: {torch.sum(atlas>0)/(atlas.shape[0]*atlas.shape[1]*atlas.shape[2])*100:.4f}%")
        images[i,0] = atlas

        # MATTER SEGMENTATION RESIZING
        seg = torch.from_numpy(mat_contents["seg_flair"]).long()
        print(f"GM coverage before: {torch.sum(seg==gm_val)/(seg.shape[0]*seg.shape[1]*seg.shape[2])*100:.4f}%")
        print(f"WM coverage before: {(torch.sum(seg==wm_val)+torch.sum(seg==wmh_val))/(seg.shape[0]*seg.shape[1]*seg.shape[2])*100:.4f}%")

        seg = torch.stack([seg==0, seg==csf_val, seg==gm_val, seg==wm_val, seg==wmh_val]).float().unsqueeze(0)
        seg = F.interpolate(seg, size=graph_size, mode="trilinear", align_corners=False).squeeze(0)
        seg = torch.argmax(seg, dim=0)

        for m, val in enumerate([0, csf_val, gm_val, wm_val, wmh_val]):
          seg[seg==m] = val

        print(f"GM coverage after: {torch.sum(seg==gm_val)/(seg.shape[0]*seg.shape[1]*seg.shape[2])*100:.4f}%")
        print(f"WM coverage after: {(torch.sum(seg==wm_val)+torch.sum(seg==wmh_val))/(seg.shape[0]*seg.shape[1]*seg.shape[2])*100:.4f}%")
        images[i,1] = seg

      torch.save(images, filename)
      print(f"Loaded images {filename}")
      return images

## GENERATE_GRAPHS
Main inputs are specified in setup. Segments GM into GM nodes based on atlas, and WM and WMH into WM nodes using solid angles and distance from center. If there are 66 atlas ROIs, an index of 5 ideally refers to roughly the same anatomical GM region based on the atlas across subjects, and an index of 100 ideally refers to roughly the same solid angle within WM across all subjects.

Takes in file directory information and graph generation parameters, and outputs and saves:
- index_maps [graph_size]: maps each voxel to a node, starting at 0. -1 indicates unmapped nodes
- adjacency_lists [num_subjects, num_nodes, variable]: lists neighbors for each node. Same across subjects. Eg, adjacency_lists[_,5] is a list of nodes neighboring node 5
- edge_weights_list [num_subjects, num_nodes, variable]: edge weights based on number of voxel-to-voxel contact points between nodes, may differ across subjects
- shell_map: maps nodes to shells, with shell 0 being GM


In [ ]:
def generate_graphs(data_directory=data_directory, output_directory=graph_directory, dataname='graph', data_table=data_table_filename, subject_ids=subject_ids,
                    graph_size=graph_size, csf_val=csf_val, wm_val=wm_val, wmh_val=wmh_val, gm_val=gm_val, num_rois=num_rois,
                    boundary_type=boundary_type, min_contact=5, num_shells=3, min_voxels=5, device=torch.device('cpu')
                    shell_ang_sectors=[7,8,9], boundaries=boundaries, neighbor_length=5, restart=False):

    num_subjects = len(subject_ids)
    images = generate_or_load_images_for_graphs(data_directory=data_directory, ubject_ids=subject_ids, graph_size=graph_size, data_table=data_table, restart=restart).to(device)

    atlases = images[:,0]
    segs = images[:,1]

    # compute number of nodes
    spatial_bins = [] #node indexes
    shell_map = [0 for _ in range(num_rois)] #maps node index to shell (gm is shell 0)

    for d in range(num_shells):
        sectors_for_this_shell = shell_ang_sectors[d]
        total_sectors_in_shell = sectors_for_this_shell * sectors_for_this_shell
        for a in range(total_sectors_in_shell):
            spatial_bins.append((d, a))
            shell_map.append(d+1)

    num_wm_slots = len(spatial_bins)
    max_nodes = num_rois + num_wm_slots
    node_tracker = torch.zeros([num_subjects, max_nodes])
    matter_tracker = torch.zeros([num_subjects, 2])

    # initialize index_maps (-1 means unmapped)
    index_maps = torch.full([num_subjects, *graph_size], -1, dtype=torch.long, device=device)


    # 1 voxel gm neighborhood --> gm nodes not segmented into an atlas ROI are assigned nearest ROI
    gm_neighborhood = torch.tensor([p for p in itertools.product([-1, 0, 1], repeat=3)], device=device)

    # neighborhood for edges
    neighbor_ops = [p for p in itertools.product(list(range(-1*neighbor_length, neighbor_length)), repeat=3)]
    neighbor_ops.remove((0,0,0))
    neighbor_ops = torch.LongTensor(neighbor_ops, device=device)

    # initializes a counter for number of "contacts" (voxel-to-voxel edges) between nodes, across dataset and per subject
    sub_interface_counts = [defaultdict(int) for _ in range(num_subjects)]
    interface_counts = defaultdict(int)

    # segmentation
    for sub in range(num_subjects):
        seg = segs[sub]
        atlas = atlases[sub]

        # compute center of mass as center of WM + WMH --> cx, cy, cz
        brain_mask = (seg == wm_val) | (seg == wmh_val)
        brain_mask_np = brain_mask.cpu().numpy()
        center = ndimage.center_of_mass(brain_mask_np)
        cx, cy, cz = center

        # compute Euclidean distance of each WM/WMH voxel from the center --> wm_distances
        dist_map_np = torch.ones(graph_size)
        dist_map_np[int(cx), int(cy), int(cz)] = 0
        dist_map_np = dist_map_np.cpu().numpy()
        dist_map_np = ndimage.distance_transform_edt(dist_map_np)
        wm_mask_np = ((seg_flair == wm_val) | (seg_flair == wmh_val)).cpu().numpy()
        wm_distances = dist_map_np[wm_mask_np]

        # assign depth thresholds for shells
        if boundaries is None:
          boundaries = np.linspace(0, 100, num_shells + 1)

        if boundary_type == "quantile": #compute depth thresholds based on distribution of wm_distances
          depth_thresholds = np.percentile(wm_distances, boundaries)
        else:
          depth_thresholds =  np.min(wm_distances)+np.array(boundaries)/100*(np.max(wm_distances)-np.min(wm_distances)) #compute depth thresholds based on percnetage of max
        depth_thresholds[-1] += 1e-5

        # use atlas to map GM voxels to nodes
        gm_coords = torch.nonzero((seg_flair == gm_val), as_tuple=False)
        for coord in gm_coords:
            gm_roi = int(atlas[coord[0], coord[1], coord[2]].item())
            # if not mapped to an ROI in atlas, assign ROI of nearest GM voxel (keep unmapped if NA)
            if gm_roi == 0:
                neighbors = []
                for op in gm_neighborhood:
                    n_coord = coord + op
                    if torch.any(n_coord < 0) or torch.any(n_coord >= torch.tensor(graph_size, device=device)):
                        continue
                    neighbors.append(int(atlas[n_coord[0], n_coord[1], n_coord[2]].item()))
                neighbors = [n for n in neighbors if n > 0]
                if neighbors:
                  gm_roi = int(random.choice(neighbors))

            if 1 <= gm_roi <= num_rois: #assign GM voxel to node. eg ROI 1 --> node 0. update trackers
                index_maps[sub, coord[0], coord[1], coord[2]] = gm_roi - 1
                node_tracker[sub, gm_roi-1] += 1
                matter_tracker[sub, 0] += 1

        # use wm_distance to map WM + WMH voxels
        wm_coords = torch.nonzero(torch.from_numpy(wm_mask_np).to(device), as_tuple=False)
        for coord in wm_coords:
            x, y, z = coord[0].item(), coord[1].item(), coord[2].item()
            voxel_dist = dist_map_np[x, y, z]

            # find depth layer
            depth_bin = 0
            for d in range(num_shells):
                if depth_thresholds[d] <= voxel_dist < depth_thresholds[d+1]:
                    depth_bin = d
                    break

            # Look up how many resolution divisions are assigned to this specific shell layer
            sectors_for_this_shell = shell_ang_sectors[depth_bin]

            # Compute Angular Sector relative to center of mass
            dx, dy, dz = x - cx, y - cy, z - cz
            azimuth = np.arctan2(dy, dx) + np.pi     # [0, 2*pi]
            elevation = np.arccos(dz / (np.sqrt(dx**2 + dy**2 + dz**2) + 1e-6)) # [0, pi]

            # Dynamically bin based on the shell's unique resolution target
            az_bin = int((azimuth / (2 * np.pi)) * sectors_for_this_shell)
            el_bin = int((elevation / np.pi) * sectors_for_this_shell)

            az_bin = min(max(az_bin, 0), sectors_for_this_shell - 1)
            el_bin = min(max(el_bin, 0), sectors_for_this_shell - 1)

            ang_bin = az_bin * sectors_for_this_shell + el_bin

            # Construct consistent Master Global Node ID mapping
            sl_idx = spatial_bins.index((depth_bin, ang_bin))
            global_node_id = num_rois + sl_idx
            index_maps[sub, x, y, z] = global_node_id

            node_tracker[sub, global_node_id] += 1
            matter_tracker[sub, 1] += 1

        # track voxel-to-voxel contacts between nodes
        valid_coords = torch.nonzero(index_maps[sub] >= 0, as_tuple=False)
        for coord in valid_coords:
            node_u = index_maps[sub, coord[0], coord[1], coord[2]].item()
            for op in neighborhood_ops:
                n_coord = coord + op
                if torch.any(n_coord < 0) or torch.any(n_coord >= torch.tensor(graph_size, device=device)):
                    continue
                node_v = index_maps[sub, n_coord[0], n_coord[1], n_coord[2]].item()
                if node_v == -1 or node_u == node_v or (node_u < num_rois and node_v < num_rois):
                    continue

                pair = tuple(sorted((node_u, node_v)))
                interface_counts[pair] += 1
                if diff_weights:
                  sub_interface_counts[sub][pair] += 1

    # pruning too small nodes (if average number of voxels less than min_voxels, or any subject has voxels less than half min_voxels)
    node_mapping = {} #maps old node index to new node index
    for r in range(num_rois):
        node_mapping[r] = r #map the gm rois to gm roi

    tracker_ids = list(range(0,num_rois)) #mask of valid nodes after pruning
    print(f"Initial number of nodes: {max_nodes}")
    node_size_list = torch.mean(node_tracker, dim=0)
    num_nodes = num_rois
    for n_val in range(num_rois, max_nodes):
        if node_size_list[n_val] < min_voxels or torch.any(node_tracker[:,n_val]<0.5*min_voxels):
            node_mapping[n_val] = -1 #prune out node
        else:
            node_mapping[n_val] = num_nodes
            tracker_ids.append(n_val)
            num_nodes += 1

    print(f"New number of nodes: {num_nodes}")

    # update index_map to fit old-->new mappings
    for old_idx, new_idx in node_mapping.items():
        if new_idx != -1:
            index_maps[index_maps == old_idx] = new_idx

    index_maps[index_maps>=num_nodes] = -1


    # unmapped WM nodes assigned to nearest WM node (if within neighbor_length)
    for sub in range(num_subjects):
        seg = segs[sub]
        wm_mask = (seg == wm_val) | (seg == wmh_val)
        unmapped_wm = wm_mask & (index_maps[sub] == -1) #get unmapped WM voxels
        mapped_nodes = index_maps[sub] >= num_rois #get mapped WM voxels

        if unmapped_wm.any() and mapped_nodes.any():
            mapped_nodes_np = mapped_nodes.cpu().numpy()

            # Compute distances and nearest mapped node coordinates
            distances, nearest_indices = ndimage.distance_transform_edt(
                ~mapped_nodes_np, return_distances=True, return_indices=True
            )

            idx_map_np = index_maps[sub].cpu().numpy()
            unmapped_wm_np = unmapped_wm.cpu().numpy()

            # Filter unmapped voxels within distance < neighbor_length
            valid_merge_mask = unmapped_wm_np & (distances < neighbor_length)

            if np.any(valid_merge_mask):
                nearest_node_ids = idx_map_np[nearest_indices[0], nearest_indices[1], nearest_indices[2]]
                idx_map_np[valid_merge_mask] = nearest_node_ids[valid_merge_mask]
                index_maps[sub] = torch.from_numpy(idx_map_np).to(device)

    # recount number of voxels per nodes AFTER pruning
    node_tracker = torch.zeros([num_subjects, num_nodes], device=device)
    for sub in range(num_subjects):
        valid_nodes = index_maps[sub][index_maps[sub] >= 0]
        counts = torch.bincount(valid_nodes, minlength=num_nodes)
        node_tracker[sub] = counts.float()

    shell_map = torch.tensor(shell_map, device=device)
    shell_map = shell_map[tracker_ids]
    node_size_list = torch.mean(node_tracker, dim=0)

    # print stats
    print("Mapping nodes to shell:")
    print(shell_map)
    gm_coverage = torch.sum(node_tracker[:,:num_rois], dim=1)/matter_tracker[:,0]*100
    wm_coverage = torch.sum(node_tracker[:,num_rois:], dim=1)/matter_tracker[:,1]*100
    print(f"Average GM coverage = {torch.mean(gm_coverage):.4f} +- {torch.std(gm_coverage):.4f}%")
    print(f"Average WM coverage = {torch.mean(wm_coverage):.4f} +- {torch.std(wm_coverage):.4f}%")
    print(f"Average GM size = {torch.mean(node_tracker[:num_rois]):.4f} +- {torch.std(node_size_list[:num_rois]):.4f}")
    print(f"Average WM size = {torch.mean(node_size_list[num_rois:]):.4f} +- {torch.std(node_size_list[num_rois:]):.4f}")
    print(f"Average node size = {torch.mean(node_size_list):.4f} +- {torch.std(node_size_list):.4f}")

    # compute edges based on number of contacts


    # generate adjacency_list
    harmonized_adjacency = [set() for n in range(num_nodes)]; counter = 0
    for (old_u, old_v), count in interface_counts.items():
        counter += 2
        new_u = node_mapping.get(old_u, -1)
        new_v = node_mapping.get(old_v, -1)
        if new_u == -1 or new_v == -1:
            continue

        harmonized_adjacency[new_u].add(new_v)
        harmonized_adjacency[new_v].add(new_u)

    print(f"Previous number of edges: {counter+num_rois}")
    final_adjacency_list = [sorted(list(edges)) for edges in harmonized_adjacency] #adjacency_list is a list of lists. list A lists neighboring nodes for node A. same for every subject
    for node in range(num_nodes):
      final_adjacency_list[node].append(node) #self loop
    adjacency_lists = [final_adjacency_list for _ in range(num_subjects)] #repeat final_adjacency_list for each subject

    degrees = [len(nodes) for nodes in final_adjacency_list]
    print(degrees)
    print(f"New number of edges: {np.sum(degrees)}")
    print(f"Average edge degree: {np.mean(degrees)} +- {np.std(degrees)}")

    # compute edge weights based on voxel-to-voxel contact points
    edge_weight_lists = []
    for sub in range(num_subjects):
        # Create an edge weight list structure unique to this subject
        sub_edge_weight_list = [[] for _ in final_adjacency_list]

        for u in range(num_nodes): #0 --> num_nodes
            u_deg = degrees[u]
            u_edge_list = np.zeros([u_deg,1]).flatten() #defaults to 0
            for ind, v in enumerate(final_adjacency_list[u]):

                if u==v: #if the nodes are the same
                  continue

                # Look up raw contact counts using old indexing to cross-reference tracked coordinates
                # We map the final new indices back to the tracked old edge dictionary keys
                old_u_lookup = [old for old, new in node_mapping.items() if new == u] #old, new
                old_v_lookup = [old for old, new in node_mapping.items() if new == v]

                if old_u_lookup and old_v_lookup:
                    ou = old_u_lookup[0]
                    ov = old_v_lookup[0]
                    edge_key = (min(ou, ov), max(ou, ov))
                    contact_points = sub_interface_counts[sub].get(edge_key, 0)

                if contact_points < min_contact: #prune out edges with too few contact points in specific subject
                  continue
                else:
                  contact_scaler = contact_points

                final_weight = contact_scaler
                u_edge_list[ind]=final_weight #if too few contacts, defaults to 0

            u_edge_list[-1] = np.max(u_edge_list) #self loop assigned max contact
            temp = np.sum(u_edge_list)
            if temp>0:
              u_edge_list = u_edge_list/np.sum(u_edge_list) #normalize
            else:
              print(f"Subject {sub} doesn't have edges for {u} node")
              # u_edge_list = 0.5*np.ones_like(u_edge_list)/u_deg #same edge weight assigned
              u_edge_list[-1] = 1.0 #self loop assigned weight of 1.0

            u_edge_list = u_edge_list.tolist()
            sub_edge_weight_list[u] = u_edge_list
        edge_weight_lists.append(sub_edge_weight_list)



    #save results
    torch.save(index_maps.cpu(), os.path.join(output_directory, f"index_maps_{dataname}.pt"))
    torch.save(adjacency_lists, os.path.join(output_directory, f"adjacency_lists_{dataname}.pt"))
    torch.save(edge_weight_lists, os.path.join(output_directory, f"edge_weight_lists_{dataname}.pt"))
    torch.save(node_tracker, os.path.join(output_directory, f"node_volumes_{dataname}.pt"))
    torch.save(shell_map, os.path.join(output_directory, f"shell_map_{dataname}.pt"))
    print("Balanced harmonized graphs generated successfully!")

    return index_maps, adjacency_lists, node_tracker, edge_weight_lists, shell_map


index_maps, adjacency_lists, node_tracker, edge_weight_lists, shell_map = generate_graphs(data_directory=data_directory, output_directory=graph_directory, dataname=dataname,
                                                                                          data_table=data_table_filename, subject_ids=subject_ids
                                                        graph_size=graph_size, neighbor_length=neighbor_length, min_contact=min_contact, min_voxels=min_voxels,
                                                        temp_type=torch.float16, num_shells=num_shells, boundary_type=boundary_type, boundaries=boundaries, shell_ang_sectors=shell_ang_sectors,
                                                        device=device)


## Check results
Check results using a 3D viewer similar to FSL

In [ ]:
def plot_graphs(graph_directory=graph_directory, dataname=dataname, index_map_name="index_maps_", edge_weight_name="edge_weight_lists_", graph_size=graph_size,
                subjects=[0], num_rois=num_rois):
    # Create a custom colormap
    index_maps = torch.load(graph_directory+index_map_name+dataname+".pt")
    edge_weight_lists = torch.load(graph_directory+edge_weight_name+dataname+".pt")

    num_nodes_list = []; gm_size_list = []; wm_size_list = []; num_edges_list = []
    for index_map, edge_list in zip(index_maps, edge_weight_lists):
      num_nodes = len(edge_list)
      # print(num_nodes)
      num_nodes_list.append(num_nodes)
      for i in range(num_rois):
        gm_size_list.append(torch.sum((index_map==i).flatten()))
        num_edges_list.append(np.sum(np.array(edge_list[i])>1e-5))
      for i in range(num_rois, num_nodes, 1):
        wm_size_list.append(torch.sum((index_map==i).flatten()))
        num_edges_list.append(np.sum(np.array(edge_list[i])>1e-5))


    plt.figure(figsize=(20,10))

    combined_data = np.hstack((gm_size_list, wm_size_list))
    bin_edges = np.histogram(combined_data, bins=30)[1].astype(np.int16)
    plt.hist(gm_size_list, bins=bin_edges, alpha=0.5, label="gm nodes")
    plt.hist(wm_size_list, bins=bin_edges, alpha=0.5, label="wm nodes")
    plt.title(f"Histogram of number of voxels per node", fontsize=20)
    plt.xlabel("Number of voxels", fontsize=20)
    plt.ylabel("Number of nodes across all subjects", fontsize=20)
    plt.legend(fontsize=20)
    # plt.xticks(bin_edges[list(range(0,len(bin_edges),5))], fontsize=14)
    plt.xticks(list(range(0,bin_edges[-1],200)), fontsize=20)
    plt.yticks(fontsize=20)

    plt.show()

    for sub in subjects:
      index_map = index_maps[sub]
            #viewing data
      max_val  = max(index_map.flatten())

      pseudorandom_colors = []

      for val in range(max_val + 1):
          # 1. Create a hash-based seed (0.0 to 1.0) for variety within the group
          hash_val = int(hashlib.md5(str(val).encode()).hexdigest(), 16)
          sub_variety = (hash_val % 1000) / 1000.0

          if 0 <= val <= num_rois-1:
              # GRAY MATTER: Warm colors (Red/Orange/Yellow)
              # Hue range: 0.0 (Red) to 0.15 (Yellow)
              h = 0.0 + (sub_variety * 0.15)
              s = 0.7 + (sub_variety * 0.3)  # High saturation
              v = 0.8 + (sub_variety * 0.2)  # High brightness
              alpha = 1.0

          elif num_rois <= val:
              # WHITE MATTER: Cool colors (Blue/Cyan/Purple)
              # Hue range: 0.5 (Cyan) to 0.7 (Deep Blue)
              h = 0.5 + (sub_variety * 0.25)
              s = 0.5 + (sub_variety * 0.25)  # Slightly more muted
              v = 0.7 + (sub_variety * 0.3)
              alpha = 0.8  # Keep your transparency requirement

          # 2. Convert HSV to RGB
          rgb = list(colorsys.hsv_to_rgb(h, s, v))
          rgb.append(alpha) # Add Alpha channel
          pseudorandom_colors.append(rgb)

      # 3. Final colormap construction
      white_bg = np.array([[1.0, 1.0, 1.0, 1.0]]) # Background for -1
      all_colors = np.vstack((white_bg, np.array(pseudorandom_colors)))
      custom_cmap = mcolors.ListedColormap(all_colors)

      # 4. Normalization
      bounds = np.arange(-1.5, max_val + 0.5, 1)
      norm = mcolors.BoundaryNorm(bounds, custom_cmap.N)

      def plot_slice(slice_idx, dir=0, data=None):

          plt.figure(figsize=(8, 6))
          # 'data' is your tensor variable
          if dir==0:
            plt.imshow(data[slice_idx,:,:].cpu().numpy(), cmap=custom_cmap, norm=norm)
          elif dir == 1:
            plt.imshow(data[:,slice_idx,:].cpu().numpy(), cmap=custom_cmap, norm=norm)
          else:
            plt.imshow(data[:,:,slice_idx].cpu().numpy(), cmap=custom_cmap, norm=norm)
          plt.colorbar(label="Intensity")
          plt.title(f"Viewing Slice: {slice_idx}")
          plt.show()

      # This creates the dynamic slider
      interact(lambda slice_idx: plot_slice(slice_idx, dir=0, data=index_map), slice_idx=(0, index_map.shape[0] - 1))

      # This creates the dynamic slider
      interact(lambda slice_idx: plot_slice(slice_idx, dir=1, data=index_map), slice_idx=(0, index_map.shape[1] - 1))

      # This creates the dynamic slider
      interact(lambda slice_idx: plot_slice(slice_idx, dir=2, data=index_map), slice_idx=(0, index_map.shape[2] - 1))

      num_voxels = []; num_edges = []; c = 0
      print(f"Subject {sub}")
      for node,lst in enumerate(edge_weight_lists[sub]):
          num_voxel = torch.sum((index_map==node).flatten()).item()
          num_voxels.append(num_voxel)
          num_edges.append(np.sum(np.array(lst)>1e-5))
          print(f"Node {node}: Num voxels = {num_voxel}, Num edges = {np.sum(np.array(lst)>1e-5)}")

plot_graphs(graph_directory=graph_directory, dataname=dataname, graph_size=graph_size,
                subjects=[0,1])
